# 17-620 Lab 2 — How Do We Compare Classical and Quantum Data?

**80 minutes | PennyLane | Fall 2026**

### The question driving this lab

**How does the way we represent data change what it means for two examples to be similar?**

We begin with classical vectors, compare their directions, then encode data into quantum states and measure their overlap. We compare two encoding methods before asking what an SVM would actually learn.

**Learning goals:** (1) explain cosine similarity and squared overlap; (2) implement angle and amplitude encoding; (3) build and compare two quantum kernel matrices; (4) estimate a kernel from measurement samples; (5) distinguish a fixed quantum feature map from a learned SVM decision boundary.

**Reading:** *A Practical Guide to Quantum Machine Learning and Quantum Optimization*, Chapter 9.

Run cells in order. **Worked example** means run and inspect; **TODO** means write code. Uncomment checks after completing each exercise. Write short answers to conceptual questions.

> A quantum circuit can compute a similarity without learning from labels. Whether that similarity helps a classifier is a separate question.

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

print("PennyLane", qml.__version__)

## Part 1 — What does classical similarity measure? (10 minutes)

Two vectors can point together, at right angles, or in opposite directions. **Cosine similarity** captures this orientation without depending on their lengths:

$$
\operatorname{cosSim}(a,b)=\frac{a\cdot b}{\|a\|\|b\|}.
$$

For nonzero vectors, **+1** means same direction, **0** means orthogonal, and **−1** means opposite directions.

First inspect the heatmap for our four example vectors. Each cell compares a **pair of data points**, not two individual features. What does the entry for points 0 and 3 tell you?

In [ ]:
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [-1.0, 0.0]])
cosine = cosine_similarity(X)
print("Classical cosine similarity:\n", np.round(cosine, 3))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cosine, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(X)), labels=[f"x{i}" for i in range(len(X))])
ax.set_yticks(range(len(X)), labels=[f"x{i}" for i in range(len(X))])
ax.set_title("Classical cosine similarity")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.show()

### Explore classical geometry

Move the four component sliders. Watch the two vectors, their dot product, their lengths, their included angle, and cosine similarity.

$$
a\cdot b=a_xb_x+a_yb_y.
$$

Try parallel, perpendicular, and opposite vectors. Then make one vector zero. Its cosine similarity becomes **undefined**, because we cannot divide by a zero-length vector.

**Question 1.** Why can cosine similarity be negative? Which pairs in the earlier heatmap are orthogonal?

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def classical_cosine_explorer(a_x=4.0, a_y=1.0, b_x=-1.0, b_y=3.0):
    a = np.array([a_x, a_y], dtype=float)
    b = np.array([b_x, b_y], dtype=float)
    dot = float(np.dot(a, b))
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    valid = norm_a > 1e-12 and norm_b > 1e-12
    cosine = float(np.clip(dot / (norm_a * norm_b), -1, 1)) if valid else None
    theta = float(np.degrees(np.arccos(cosine))) if valid else None

    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    for v, label, color in [(a, "a", "tab:blue"), (b, "b", "tab:green")]:
        ax.annotate("", xy=tuple(v), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=2.8, mutation_scale=18))
        if np.linalg.norm(v) > 1e-12:
            ax.text(v[0] * 1.08 + 0.1, v[1] * 1.08 + 0.1, label,
                    fontsize=14, color=color, weight="bold")
    ax.axhline(0, color="gray", lw=0.9)
    ax.axvline(0, color="gray", lw=0.9)
    ax.set(xlim=(-7, 7), ylim=(-7, 7), aspect="equal",
           xlabel="x component", ylabel="y component",
           title="Two classical feature vectors")
    ax.grid(alpha=0.22)
    plt.show()

    print(f"Dot product: a·b = ({a_x:g})({b_x:g}) + ({a_y:g})({b_y:g}) = {dot:.3f}")
    print(f"Vector magnitudes: ||a|| = {norm_a:.3f}, ||b|| = {norm_b:.3f}")
    if not valid:
        print("Cosine similarity: undefined (at least one vector is zero).")
        return
    if np.isclose(cosine, 1.0, atol=1e-10):
        direction = "same direction"
    elif np.isclose(cosine, -1.0, atol=1e-10):
        direction = "opposite directions"
    elif np.isclose(cosine, 0.0, atol=1e-10):
        direction = "orthogonal"
    elif cosine > 0:
        direction = "acute angle / positive similarity"
    else:
        direction = "obtuse angle / negative similarity"
    print(f"cosSim(a,b) = {cosine:+.3f} | Angle = {theta:.1f}° | {direction}")

component_controls = {
    "a_x": widgets.FloatSlider(value=4, min=-6, max=6, step=0.5, description="aₓ",
                               continuous_update=False),
    "a_y": widgets.FloatSlider(value=1, min=-6, max=6, step=0.5, description="aᵧ",
                               continuous_update=False),
    "b_x": widgets.FloatSlider(value=-1, min=-6, max=6, step=0.5, description="bₓ",
                               continuous_update=False),
    "b_y": widgets.FloatSlider(value=3, min=-6, max=6, step=0.5, description="bᵧ",
                               continuous_update=False),
}
component_output = widgets.interactive_output(classical_cosine_explorer, component_controls)
display(widgets.VBox([widgets.HTML("<b>Drag to change vector components</b>"),
                      *component_controls.values(), component_output]))

## Part 2 — Why do we call it **squared overlap**?

Quantum states are vectors too, although their components may be complex. To compare states, first take their **inner product**:

$$
s(x,y)=\langle\phi(y)|\phi(x)\rangle.
$$

This **overlap amplitude** can be negative or complex. A measurement probability cannot. The **Born rule** turns the amplitude into a probability by taking its **squared magnitude**:

$$
\boxed{K_Q(x,y)=|\langle\phi(y)|\phi(x)\rangle|^2.}
$$

We square the **magnitude of the inner product**, not each vector component. For example, an amplitude of \(0.6+0.3i\) gives probability \(0.6^2+0.3^2=0.45\). For pure quantum states, squared overlap is also called **fidelity**.

### How do we measure it?

Prepare the quantum state for x using a circuit \(U(x)\). Then undo the preparation for y using \(U(y)^\dagger\). The amplitude of returning to all zeros is

$$
\langle0\ldots0|U(y)^\dagger U(x)|0\ldots0\rangle=\langle\phi(y)|\phi(x)\rangle.
$$

By the Born rule, the **probability** of measuring all zeros is its squared magnitude. This is the measurement we will implement below.

### The connection to cosine similarity

If we load **nonzero, real classical vectors** into normalized quantum-state amplitudes, their inner product equals classical cosine similarity. Thus

$$
K_Q(x,y)=\operatorname{cosSim}(x,y)^2.
$$

Opposite real vectors then have squared overlap 1 because a global minus sign does not distinguish quantum states. **Other quantum encodings generally define different kernels.**

**Question 2.** If the overlap amplitude is \(-0.8\), what measurement probability results, and why?

### See what squaring changes

Move the angle slider. The teal vector stays fixed, while the orange vector rotates. **Cosine similarity can be negative**; its square cannot.

At 180°, the vectors point in opposite directions: cosine similarity is −1 and squared overlap is +1. This demonstrates the relationship for **real amplitude encoding only**, not for all quantum feature maps.

In [ ]:
import ipywidgets as widgets

def explore_similarity(angle=60):
    theta = np.deg2rad(angle)
    x = np.array([1., 0.])
    y = np.array([np.cos(theta), np.sin(theta)])
    cosine = float(x @ y)
    fidelity = cosine**2
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.arrow(0, 0, x[0], x[1], color="teal", width=0.01, head_width=0.07, length_includes_head=True)
    ax.arrow(0, 0, y[0], y[1], color="darkorange", width=0.01, head_width=0.07, length_includes_head=True)
    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal", xlabel="Feature 1",
           ylabel="Feature 2", title=f"Angle {angle}° | cosine {cosine:+.3f} | squared overlap {fidelity:.3f}")
    ax.grid(alpha=0.25)
    plt.show()

widgets.interact(explore_similarity, angle=widgets.IntSlider(value=60, min=0, max=180, step=5, continuous_update=False));

## Part 3 — Encode data with angles and build a quantum kernel (40 minutes)

**First question: How does a classical number become a quantum state?**

### Worked example: one number controls one rotation

An angle-encoding circuit uses an input number as a rotation parameter:

$$
R_Y(\theta)|0\rangle=\cos(\theta/2)|0\rangle+\sin(\theta/2)|1\rangle.
$$

For \(\theta=0,\pi/2,\pi\), the probabilities of measuring 0 are 1, 1/2, and 0. Run the complete example below and inspect the circuit drawing.

A gate's **rotation angle** is not automatically the geometric angle between two classical feature vectors.

In [ ]:
worked_dev = qml.device("default.qubit", wires=1)

@qml.qnode(worked_dev)
def worked_rotation(theta):
    qml.RY(theta, wires=0)
    return qml.probs(wires=[0])

for theta in [0.0, np.pi / 2, np.pi]:
    print(f"theta={theta:.3f}: P(0),P(1)={worked_rotation(theta)}")
print(qml.draw(worked_rotation)(np.pi / 2))

### Exercise 3A — Put two classical numbers into two qubits

Give each qubit a Y rotation using one of the two features. Implement the TODO in the next cell using PennyLane's AngleEmbedding.

Return the measurement probabilities in order **00, 01, 10, 11**, then enable the checks.

**Question:** Why might raw features need to be rescaled as angles? What happens to measurement probabilities if an input angle changes by \(2\pi\)?

In [ ]:
angle_dev = qml.device("default.qubit", wires=2)

@qml.qnode(angle_dev)
def angle_state(x):
    # TODO: replace pass with qml.AngleEmbedding(features=x, wires=[0, 1], rotation="Y")
    pass
    return qml.probs(wires=[0, 1])

angle_inputs = np.array([[0.0, 0.0], [np.pi/2, 0.0],
                         [0.0, np.pi/2], [np.pi/2, np.pi/2]])

print("Example:", angle_state(angle_inputs[0]))
print(qml.draw(angle_state)(angle_inputs[1]))
# Self-check: uncomment after implementing
# assert np.allclose(angle_state(angle_inputs[0]), [1, 0, 0, 0])
# assert np.allclose(angle_state(angle_inputs[3]), [0.25, 0.25, 0.25, 0.25])

### Exercise 3B — Use squared overlap to build an **angle kernel**

**Second question: How do we compare two encoded states?**

1. Prepare the state for x with AngleEmbedding.
2. Undo the encoding for y using its adjoint.
3. Measure the probability of returning to **|00⟩**. This is the **squared overlap**.
4. Evaluate that circuit on every pair of points to build the **angle quantum kernel matrix**.
5. Display a heatmap and enable the self-checks.

The circuit measures **a probability**, not the original signed cosine similarity.

In [ ]:
kernel_dev = qml.device("default.qubit", wires=2)

@qml.qnode(kernel_dev)
def angle_overlap(x, y):
    # TODO: qml.AngleEmbedding for x
    # TODO: qml.adjoint(qml.AngleEmbedding) for y
    return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

def make_kernel(points, circuit):
    return np.array([[circuit(x, y) for y in points] for x in points])

# TODO: uncomment once the function is implemented
# K_angle = make_kernel(angle_inputs, angle_overlap)
# print(np.round(K_angle, 3))
# fig, ax = plt.subplots(figsize=(5, 4))
# im = ax.imshow(K_angle, vmin=0, vmax=1, cmap="viridis")
# ax.set_xticks(range(4), labels=[f"x{i}" for i in range(4)])
# ax.set_yticks(range(4), labels=[f"x{i}" for i in range(4)])
# ax.set_title("Quantum angle-overlap kernel")
# fig.colorbar(im, ax=ax, label="P(00)")
# plt.show()
# assert np.allclose(K_angle, K_angle.T, atol=1e-8)
# assert np.allclose(np.diag(K_angle), np.ones(4), atol=1e-8)
# assert np.all((K_angle >= -1e-8) & (K_angle <= 1 + 1e-8))
# assert np.isclose(angle_overlap(angle_inputs[0], angle_inputs[1]), 0.5)

### One word, two different similarity functions

Our quantum kernel depends on **how we encoded the data**. For one-qubit Y-angle encoding with rotation parameters a and b,

$$
K_{\mathrm{angle}}(a,b)=\cos^2((a-b)/2).
$$

For real amplitude-encoded vectors separated by a **geometric angle** \(\theta\), the kernel is \(\cos^2\theta\).

The comparison plot uses the same numerical angle on the horizontal axis, but it means a **gate-parameter difference** for angle encoding and a **vector angle** for amplitude encoding.

**Question 3.** At a numerical angle of 180°, what does each curve predict? What does this teach you about feature maps?

In [ ]:
delta = np.linspace(0, np.pi, 361)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.degrees(delta), np.cos(delta), label="Cosine similarity")
ax.plot(np.degrees(delta), np.cos(delta)**2, label="Amplitude kernel: cos²(delta)")
ax.plot(np.degrees(delta), np.cos(delta/2)**2, label="RY angle kernel: cos²(delta/2)")
ax.set(xlabel="Angle / rotation difference (degrees)", ylabel="Similarity / squared overlap")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Part 4 — Change the encoding, change the quantum kernel (15 minutes)

In Part 3, features became **rotation angles**. Now the components of each real vector become the **amplitudes** of a quantum state. Four components fit into two qubits.

**Repeat exactly the kernel workflow from Part 3:**

1. Prepare the state for vector a with AmplitudeEmbedding.
2. Undo the preparation for vector b using its adjoint.
3. Measure the probability of **|00⟩**, giving the amplitude kernel entry.
4. Build the **full amplitude quantum kernel matrix** with the existing kernel-matrix helper.
5. Compare its heatmap with **classical cosine similarity squared** using the same color scale.

For nonzero real vectors the two matrices agree mathematically. But computing that classical squared cosine is easy, and amplitude-state preparation on quantum hardware can be expensive. This example does **not** show quantum advantage.

**Exercise 4.** Complete the amplitude overlap circuit and both kernel matrices. Why do a vector and its negative produce squared overlap 1?

In [ ]:
amp_dev = qml.device("default.qubit", wires=2)

@qml.qnode(amp_dev)
def amplitude_overlap(a, b):
    # TODO: qml.AmplitudeEmbedding(a, wires=[0, 1], normalize=True)
    # TODO: qml.adjoint(qml.AmplitudeEmbedding)(b, wires=[0, 1], normalize=True)
    return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

A = np.array([[1., 0., 0., 0.], [0., 1., 0., 0.],
              [1., 1., 0., 0.], [-1., 0., 0., 0.]])

# TODO: calculate the quantum AMPLITUDE kernel matrix (as in Part 3).
# K_amplitude = make_kernel(A, amplitude_overlap)
# K_classical_squared = cosine_similarity(A)**2

# TODO: uncomment and plot the matrices on an identical [0,1] color scale
# fig, axes = plt.subplots(1, 2, figsize=(10, 4))
# for ax, matrix, title in zip(axes, [K_amplitude, K_classical_squared],
#                              ["Quantum amplitude kernel", "Classical cosine²"]):
#     im = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
#     ax.set_title(title)
#     ax.set_xticks(range(4), labels=[f"a{i}" for i in range(4)])
#     ax.set_yticks(range(4), labels=[f"a{i}" for i in range(4)])
# fig.colorbar(im, ax=axes)
# plt.show()

# assert np.allclose(K_amplitude, K_classical_squared, atol=1e-7)
# assert np.isclose(K_amplitude[0, 3], 1.0)

## Part 5 — Measure the kernel using finite shots (15 minutes)

So far, our ideal simulator has returned an **exact probability**. Real measurements produce **individual bitstrings**, not perfect probabilities.

We can estimate squared overlap by repeating the circuit and counting the all-zero outcomes:

$$
\widehat K(x,y)=\frac{\text{number of 00 outcomes}}{\text{number of shots}}.
$$

This is an **estimate of the same kernel**. More shots usually reduce sampling uncertainty; they do not change the feature map.

**Exercise 5.** Complete the inverse circuit, sample 1,000 outcomes, and compare the fraction of 00 results with the exact angle kernel entry.

In [ ]:
shots = 1000
sample_dev = qml.device("default.qubit", wires=2)

@qml.set_shots(shots=shots)
@qml.qnode(sample_dev)
def sampled_angle_overlap(x, y):
    # TODO: embed x and undo the embedding of y
    return qml.sample(wires=[0, 1])

x, y = angle_inputs[0], angle_inputs[1]
# TODO: samples = sampled_angle_overlap(x, y)
# TODO: estimate = np.mean(np.all(samples == 0, axis=1))
# TODO: compare to angle_overlap(x, y); expect sampling variation
# assert samples.shape == (shots, 2)
# assert abs(estimate - angle_overlap(x, y)) < 0.12

## Part 6 — What would a quantum-kernel SVM actually learn?

**Our quantum circuits have not learned anything from labels.** We chose both feature maps in advance. Each map defines a **fixed kernel** for a pair of data points. Finite-shot estimates may fluctuate, but we are not training the encoding.

A **support vector machine (SVM)** can use a kernel matrix to learn a decision boundary. A **quantum-kernel SVM (QSVM)** can use our quantum-computed matrix with the **same classical optimizer**.

For binary labels \(y_i\in\{-1,+1\}\), a soft-margin SVM learns coefficients \(\alpha_i\) by solving

$$
\max_{\alpha}\;\sum_i\alpha_i-\frac12\sum_{i,j}\alpha_i\alpha_j y_i y_j K(x_i,x_j),
$$

subject to \(0\le\alpha_i\le C\) and \(\sum_i\alpha_i y_i=0\). **The kernel stays fixed; the SVM coefficients change.**

A different model could make the quantum feature map **trainable**, introducing parameters optimized through a separate objective. A **variational quantum classifier** trains circuit parameters to produce predictions. **Neither is part of this lab.**

**Question 6.** In a fixed-kernel QSVM, which quantities learn from the training labels? Which quantities do not?

## Exit ticket — Explain the whole story

Answer each question in a few sentences:

1. **Classical comparison:** What does cosine similarity measure, and why can it be negative?
2. **Quantum measurement:** Why do we take the **squared magnitude** of an overlap amplitude? What probability does the inverse circuit measure?
3. **Encoding:** Why do angle and amplitude encoding produce different kernels? When does squared overlap equal cosine similarity squared?
4. **Learning:** What is fixed in this lab? What would an SVM optimize? What changes when we sample with finite shots?

**Optional extension:** Pass a precomputed kernel matrix to a classical SVM using scikit-learn, and compare it against a classical kernel on the same train/test split. Classification accuracy alone does **not** establish quantum advantage.

**Submission:** Save and submit your completed notebook through the course LMS.